# Into The Eagle's Nest — Object Patcher

This notebook patches treasure, health, and lift-pass tiles from the `OBJTBL0` object table
into the four `.lvl` map files.

## Background

The original game uses **two separate data sources** to build a level:

1. **Block map** (`data/maps`) — the structural layout expanded into tiles.  
   Keys and some ammo are baked into this. Our `.lvl` files come from here.

2. **Object table** (`OBJTBL0` in the executable) — a flat list of objects with precise
   tile coordinates. This holds: paintings, vases, pendants, cold food, first aid, lift passes.

The `MAKEMP` + `PUTOBJ` routines in the Amiga ASM expand the block map first, then overlay
the object tiles on top. Our `.lvl` files were generated from step 1 only, so step 2 objects
are missing. This notebook adds them.

## Coordinate System

```
Amiga buffer / .lvl file:  row 0 = TOP of map,  Y increases DOWNWARD
SpriteKit:                 row 0 = BOTTOM,       Y increases UPWARD
```

Object X/Y in `OBJTBL0` are **direct tile coordinates** in the Amiga system (same as the
.lvl file byte layout), so no conversion is needed when patching into the file.
SpriteKit's `populateTileMap` already flips rows on load.

## Tile Placement Pattern (`PTOBCH` / `OBCHTB`)

Each object writes up to a 2×2 block (zero bytes are skipped):

```
fileRow Y:   [X] = top-left tile,   [X+1] = top-right tile
fileRow Y+1: [X] = bottom-left tile, [X+1] = bottom-right tile
```

| Type | Name       | TL    | TR    | BL    | BR    |
|------|------------|-------|-------|-------|-------|
| 0    | Ammo       | 0x60  | 0x61  | —     | —     |
| 1    | Painting   | 0x62  | —     | 0x63  | —     |
| 2    | Vase       | 0x64  | —     | 0x65  | —     |
| 4    | Cold Food  | 0x69  | —     | 0x6A  | —     |
| 6    | First Aid  | 0x6C  | —     | 0x6D  | —     |
| 7    | Pendant    | 0x6E  | —     | 0x6F  | —     |
| 8    | Lift Pass  | 0x6B  | —     | —     | —     |

In [ ]:
# ── Cell 1: Imports and configuration ─────────────────────────────────────────

from pathlib import Path
import sys

# castle1_objects.py must be in the same folder as this notebook
sys.path.insert(0, str(Path.cwd()))
from castle1_objects import CASTLE1_OBJECTS, TYPE_NAMES, LEVEL_FILES, LEVEL_NAMES

# ── Paths ─────────────────────────────────────────────────────────────────────
# INPUT:  folder containing the four .lvl files
# OUTPUT: folder where patched files will be written
# Edit these two lines to match your folder layout.

INPUT_DIR  = Path(".")          # folder containing ground.lvl etc.
OUTPUT_DIR = Path("patched")    # will be created if it doesn't exist

OUTPUT_DIR.mkdir(exist_ok=True)

# ── Map dimensions ────────────────────────────────────────────────────────────
COLS = 96
ROWS = 168
EXPECTED_SIZE = COLS * ROWS   # 16,128 bytes

print(f"Input  dir : {INPUT_DIR.resolve()}")
print(f"Output dir : {OUTPUT_DIR.resolve()}")
print(f"Expected file size: {EXPECTED_SIZE} bytes")

In [ ]:
# ── Cell 2: OBCHTB — tile lookup table ────────────────────────────────────────
#
# Transcribed from OBCHTB in the ASM (line 2419).
# Layout: 4 bytes per type = [top-left, top-right, bottom-left, bottom-right]
# A zero byte means 'skip' — PTOBCH only writes non-zero values.
#
# The four values map to positions relative to the object's (X, Y) origin:
#   top-left    → file byte at row Y,   col X
#   top-right   → file byte at row Y,   col X+1
#   bottom-left → file byte at row Y+1, col X
#   bottom-right→ file byte at row Y+1, col X+1

OBCHTB = {
#  type : (top-left, top-right, bottom-left, bottom-right)
    0:    (0x60,     0x61,      0x00,        0x00),   # Ammo
    1:    (0x62,     0x00,      0x63,        0x00),   # Painting
    2:    (0x64,     0x00,      0x65,        0x00),   # Vase
    3:    (0x00,     0x00,      0x00,        0x00),   # BUG (unused)
    4:    (0x69,     0x00,      0x6A,        0x00),   # Cold Food
    5:    (0x00,     0x00,      0x00,        0x00),   # Dynamite (unused here)
    6:    (0x6C,     0x00,      0x6D,        0x00),   # First Aid
    7:    (0x6E,     0x00,      0x6F,        0x00),   # Pendant
    8:    (0x6B,     0x00,      0x00,        0x00),   # Lift Pass
    9:    (0x00,     0x00,      0x00,        0x00),   # Key (in block map already)
}

print("OBCHTB tile table loaded.")
print(f"{'Type':<5} {'Name':<12} {'TL':>4} {'TR':>4} {'BL':>4} {'BR':>4}")
print("-" * 36)
for t, tiles in OBCHTB.items():
    tl, tr, bl, br = tiles
    print(f"{t:<5} {TYPE_NAMES[t]:<12} {tl:>4X} {tr:>4X} {bl:>4X} {br:>4X}")

In [ ]:
# ── Cell 3: Core patching function ────────────────────────────────────────────
#
# patch_object_into_level(data, obj_type, x, y)
#
# Parameters:
#   data     : bytearray — the mutable level data (96×168 bytes)
#   obj_type : int       — object type 0–9
#   x        : int       — tile column (0–95)
#   y        : int       — tile row in Amiga coords (0 = top, 167 = bottom)
#                          This matches the .lvl file byte layout directly.
#
# Returns:
#   list of (row, col, old_byte, new_byte) describing every write made,
#   for the change-log report in cell 5.

def patch_object_into_level(data: bytearray, obj_type: int, x: int, y: int) -> list:
    tiles = OBCHTB.get(obj_type, (0, 0, 0, 0))
    
    # The four positions in the 2×2 grid, as (row_offset, col_offset)
    positions = [
        (0, 0),   # top-left
        (0, 1),   # top-right
        (1, 0),   # bottom-left
        (1, 1),   # bottom-right
    ]
    
    changes = []
    
    for (row_off, col_off), tile_byte in zip(positions, tiles):
        # Skip zero tiles — PTOBCH only writes if source byte is non-zero
        if tile_byte == 0x00:
            continue
        
        row = y + row_off
        col = x + col_off
        
        # Bounds check — guard against objects placed near map edges
        if row < 0 or row >= ROWS or col < 0 or col >= COLS:
            print(f"  ⚠️  Out of bounds: type={obj_type} ({TYPE_NAMES[obj_type]}), "
                  f"x={x}, y={y}, target row={row}, col={col} — skipped")
            continue
        
        byte_index = row * COLS + col
        old_byte   = data[byte_index]
        data[byte_index] = tile_byte
        changes.append((row, col, old_byte, tile_byte))
    
    return changes


print("patch_object_into_level() defined.")

In [ ]:
# ── Cell 4: Load all four level files ─────────────────────────────────────────
#
# We load each .lvl file into a bytearray (mutable bytes — unlike Python's
# bytes type which is immutable, bytearray lets us modify individual bytes
# in-place without copying the whole thing every time).
#
# levels[i] = bytearray for level i  (0=ground, 1=basement, 2=first, 3=second)

levels = {}

for level_index, filename in enumerate(LEVEL_FILES):
    path = INPUT_DIR / filename
    if not path.exists():
        print(f"  ⚠️  Missing: {path} — skipping level {level_index}")
        continue
    
    raw = path.read_bytes()
    
    if len(raw) != EXPECTED_SIZE:
        print(f"  ❌ {filename}: unexpected size {len(raw)} "
              f"(expected {EXPECTED_SIZE}) — skipping")
        continue
    
    levels[level_index] = bytearray(raw)
    print(f"  ✅ Loaded level {level_index} ({LEVEL_NAMES[level_index]}): "
          f"{filename} — {len(raw)} bytes")

print(f"\n{len(levels)}/4 level files loaded.")

In [ ]:
# ── Cell 5: Patch all objects into their levels ────────────────────────────────
#
# Iterates through CASTLE1_OBJECTS and calls patch_object_into_level() for each.
# A detailed change log is printed so you can verify every write before
# examining the output files in a hex editor.

from collections import defaultdict

# Summary counters per level
summary = defaultdict(lambda: defaultdict(int))  # summary[level][type_name] += count
total_patches = 0
total_overwrites = 0   # how many tiles replaced a non-floor byte (worth knowing)

print("Patching objects into levels...")
print("=" * 70)

for obj_index, (obj_type, x, y, level) in enumerate(CASTLE1_OBJECTS):
    
    if level not in levels:
        print(f"  ⚠️  Object {obj_index}: level {level} not loaded — skipped")
        continue
    
    type_name = TYPE_NAMES.get(obj_type, f"type{obj_type}")
    
    # Skip types that write no tiles (BUG=3, Dynamite=5, Key=9)
    tl, tr, bl, br = OBCHTB[obj_type]
    if tl == 0 and tr == 0 and bl == 0 and br == 0:
        print(f"  — Object {obj_index:3d}: {type_name:<12} level={level}  "
              f"x={x:3d} y={y:3d}  (no tiles to write)")
        continue
    
    changes = patch_object_into_level(levels[level], obj_type, x, y)
    
    total_patches += len(changes)
    summary[level][type_name] += 1
    
    # Report each write, flagging overwrites of non-floor bytes
    overwrite_flags = []
    for (row, col, old, new) in changes:
        if old != 0x00:
            overwrite_flags.append(f"overwrite 0x{old:02X}→0x{new:02X} at ({col},{row})")
            total_overwrites += 1
    
    flag_str = "  ⚠️  " + ", ".join(overwrite_flags) if overwrite_flags else ""
    
    print(f"  ✍️  Object {obj_index:3d}: {type_name:<12} level={level}  "
          f"x=0x{x:02X}({x:3d}) y=0x{y:02X}({y:3d})  "
          f"+{len(changes)} tile(s){flag_str}")

print()
print("=" * 70)
print(f"Total tile writes : {total_patches}")
print(f"Non-floor overwrites: {total_overwrites} (check ⚠️  lines above)")

In [ ]:
# ── Cell 6: Summary by level and object type ──────────────────────────────────

print("Object counts per level after patching:")
print()

for level_index in sorted(summary.keys()):
    print(f"  Level {level_index} — {LEVEL_NAMES[level_index]}")
    for type_name, count in sorted(summary[level_index].items()):
        print(f"    {type_name:<12} × {count}")
    print()

In [ ]:
# ── Cell 7: Write patched level files ─────────────────────────────────────────
#
# Output files are named identically to the inputs but written to OUTPUT_DIR.
# Open both the original and patched files side-by-side in your hex editor
# to visually verify the new tile bytes have landed in the right places.

for level_index, filename in enumerate(LEVEL_FILES):
    if level_index not in levels:
        print(f"  ⚠️  Level {level_index} not loaded — no output file written")
        continue
    
    out_path = OUTPUT_DIR / filename
    out_path.write_bytes(levels[level_index])
    print(f"  ✅ Written: {out_path}  ({len(levels[level_index])} bytes)")

print()
print("Done. Open the original and patched .lvl files in your hex editor")
print("to compare. Object tile bytes to look for:")
print("  Painting  : 0x62, 0x63")
print("  Vase      : 0x64, 0x65")
print("  Cold Food : 0x69, 0x6A")
print("  First Aid : 0x6C, 0x6D")
print("  Pendant   : 0x6E, 0x6F")
print("  Lift Pass : 0x6B")

In [ ]:
# ── Cell 8: Sanity check — verify specific objects are present ────────────────
#
# Spot-checks a few known objects from the table to confirm they landed correctly.
# Uses the first few OBJTBL0 entries whose positions we can reason about.
#
# How bytearray indexing works:
#   The .lvl file stores data row by row, top to bottom (row 0 = top of map).
#   Byte index = row * 96 + column
#   So for an object at Amiga tile (x, y): index = y * 96 + x

def check_object(level_data: bytearray, x: int, y: int, expected_tile: int, description: str):
    """Checks one tile position and reports pass/fail."""
    byte_index = y * COLS + x
    actual = level_data[byte_index]
    status = "✅" if actual == expected_tile else "❌"
    print(f"  {status} {description}: "
          f"pos=({x},{y}) index={byte_index} "
          f"expected=0x{expected_tile:02X} got=0x{actual:02X}")

print("Spot checks (Level 0 — Ground Floor):")
if 0 in levels:
    # Lift Pass at (0x57, 0x8E) = (87, 142) — top-left tile should be 0x6B
    check_object(levels[0], 0x57, 0x8E, 0x6B, "Lift Pass top tile")
    
    # Cold Food at (0x04, 0x0F) = (4, 15) — top tile 0x69, bottom tile 0x6A
    check_object(levels[0], 0x04, 0x0F, 0x69, "Cold Food top tile")
    check_object(levels[0], 0x04, 0x10, 0x6A, "Cold Food bottom tile")
    
    # Pendant at (0x3A, 0x81) = (58, 129) — top tile 0x6E, bottom tile 0x6F
    check_object(levels[0], 0x3A, 0x81, 0x6E, "Pendant top tile")
    check_object(levels[0], 0x3A, 0x82, 0x6F, "Pendant bottom tile")
    
    # First Aid at (0x52, 0xA1) = (82, 161) — top tile 0x6C
    check_object(levels[0], 0x52, 0xA1, 0x6C, "First Aid top tile")
    check_object(levels[0], 0x52, 0xA2, 0x6D, "First Aid bottom tile")
    
    # Painting at (0x35, 0x9A) = (53, 154) — top tile 0x62
    check_object(levels[0], 0x35, 0x9A, 0x62, "Painting top tile")
    check_object(levels[0], 0x35, 0x9B, 0x63, "Painting bottom tile")

print()
print("Spot checks (Level 1 — Basement):")
if 1 in levels:
    # First Aid at (0x25, 0x86) = (37, 134)
    check_object(levels[1], 0x25, 0x86, 0x6C, "First Aid top tile")
    # Lift Pass at (0x22, 0x8F) = (34, 143)
    check_object(levels[1], 0x22, 0x8F, 0x6B, "Lift Pass tile")

In [ ]:
# ── Cell 9 (optional): ASCII map preview of object positions ──────────────────
#
# Prints a low-resolution text map of a chosen level showing where objects
# landed. Each character represents a 4×4 block of tiles.
# Useful for checking no objects ended up inside walls.

PREVIEW_LEVEL = 0   # ← change this to 0, 1, 2, or 3

# Symbol map for tile byte values in our preview
# Anything not listed shows as '.' (floor) or '#' (wall)
SYMBOLS = {
    0x00: '.',   # floor
    0x6B: 'L',   # Lift Pass
    0x62: 'P',   # Painting (top)
    0x63: 'p',   # Painting (bottom)
    0x64: 'V',   # Vase (top)
    0x65: 'v',   # Vase (bottom)
    0x69: 'F',   # Cold Food (top)
    0x6A: 'f',   # Cold Food (bottom)
    0x6C: 'H',   # First Aid (top)
    0x6D: 'h',   # First Aid (bottom)
    0x6E: 'T',   # Pendant (top)
    0x6F: 't',   # Pendant (bottom)
    0x60: 'A',   # Ammo
    0x68: 'K',   # Key
    0x2A: '^',   # Lift
}

if PREVIEW_LEVEL not in levels:
    print(f"Level {PREVIEW_LEVEL} not loaded.")
else:
    data = levels[PREVIEW_LEVEL]
    level_name = LEVEL_NAMES[PREVIEW_LEVEL]
    
    print(f"Level {PREVIEW_LEVEL} — {level_name}  (1 char = 4×4 tiles, top of map at top)")
    print(f"Legend: .=floor  #=wall  L=LiftPass  P/p=Painting  V/v=Vase")
    print(f"        F/f=ColdFood  H/h=FirstAid  T/t=Pendant  A=Ammo  K=Key  ^=Lift")
    print()
    
    # Sample every 4th row and every 4th column (= block resolution)
    preview_rows = range(0, ROWS, 4)
    preview_cols = range(0, COLS, 4)
    
    for row in preview_rows:
        line = ""
        for col in preview_cols:
            tile = data[row * COLS + col]
            if tile in SYMBOLS:
                line += SYMBOLS[tile]
            elif tile == 0x00:
                line += '.'
            else:
                line += '#'   # any wall/structural tile
        print(line)
    
    # Count objects placed on this level
    obj_on_level = [(t, x, y) for (t, x, y, lv) in CASTLE1_OBJECTS if lv == PREVIEW_LEVEL]
    print(f"\nObjects on this level: {len(obj_on_level)}")